In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = '../models/artifacts/diagnostics/severe_false_negatives.csv'

In [3]:
df_neg = pd.read_csv(data)

In [4]:
df_neg.shape

(890, 336)

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
df_neg.head(50);

In [7]:
df_neg.TransactionDT.head(25)

0     15685940
1     14837130
2     13644797
3     15286275
4     15286564
5     12963896
6     11304628
7     14350594
8     11304538
9     11305000
10    14511241
11    14677070
12    14773573
13    14080680
14    15635458
15    12523105
16    13195906
17    14910795
18    14344610
19    14344378
20    13192113
21    13914551
22    13110720
23    14925888
24    11304340
Name: TransactionDT, dtype: int64

In [8]:
# Define the explicit list of target attributes for diagnostic profiling
target_cols = [
    "isFraud", "TransactionDT", "TransactionAmt", "ProductCD", 
    "card1", "card2", "card3", "card4", "card5", "card6", 
    "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain",'C2',
]

# Extract the sub-matrix from your severe false negatives DataFrame
df_check_1 = df_neg[target_cols]

# Display the top 25 records to inspect card consistency and time bursts
df_check_1.tail(50)

,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C2
840,1,14953242,50.000,H,6957,555.0,150.0,visa,166.0,debit,441.0,87.0,-1.0,-1.0,gmail.com,gmail.com,1.0
841,1,15290635,107.950,W,14290,512.0,150.0,visa,226.0,debit,225.0,87.0,-1.0,-1.0,missing,missing,2.0
842,1,15639525,250.000,W,15066,170.0,150.0,mastercard,102.0,credit,337.0,87.0,-1.0,-1.0,gmail.com,missing,3.0
843,1,12914587,280.000,W,13478,111.0,150.0,mastercard,224.0,debit,330.0,87.0,-1.0,-1.0,gmail.com,missing,1.0
844,1,13390383,100.000,H,16075,514.0,150.0,mastercard,102.0,credit,299.0,87.0,-1.0,-1.0,yahoo.com,yahoo.com,1.0
845,1,15460163,305.000,W,17999,482.0,150.0,visa,226.0,debit,337.0,87.0,-1.0,-1.0,gmail.com,missing,4.0
846,1,15807118,59.000,W,4383,271.0,150.0,visa,226.0,credit,324.0,87.0,6.0,-1.0,aol.com,missing,1.0
847,1,12957582,150.000,R,10994,125.0,150.0,american express,-1.0,credit,472.0,87.0,-1.0,14.0,anonymous.com,anonymous.com,1.0
848,1,13799953,59.000,W,6457,321.0,150.0,visa,195.0,debit,325.0,87.0,4.0,-1.0,msn.com,missing,5.0
849,1,13385161,59.000,W,17188,321.0,150.0,visa,226.0,debit,264.0,87.0,3.0,-1.0,missing,missing,3.0


In [9]:
#round number or .950 high risk fraud
#same user high risk fraud
#deviation from email , and irregular same transactions 
#

In [14]:
# 1. Define the precise filter for this specific user profile
user_filter = (
    (df_neg['card1'] ==17188) & 
    (df_neg['card2'] == 321.0) & 
    (df_neg['card3'] == 150.0)
)

# 2. Define the explicit list of target attributes we are auditing
target_columns = [
    "isFraud", "TransactionDT", "TransactionAmt", "ProductCD", 
    "card1", "card2", "card3", "card4", "card5", "card6", 
    "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain",'C1','C2',
    'D1','D2','D3','D4','D5','D6',
    'D7', 'D8', 'D9','D10','D11','D12','D13','D14','D15','C13',

]

# 3. Extract, sort chronologically, and display the isolated user history
df_user_check = df_neg[user_filter].sort_values("TransactionDT")

# Dynamic check to ensure columns exist in your specific dataframe configuration
existing_columns = [col for col in target_columns if col in df_user_check.columns]
df_user_check = df_user_check[existing_columns]

# Display all rows for this user profile to trace their exact narrative
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
df_user_check

,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,C13
497,1,11496296,49.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,-1.0,-1.0,missing,missing,1.0,2.0,11.0,11.0,11.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,-1.0,-1.0,-1.0,11.0,1.0
261,1,11501949,58.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,41.0,-1.0,gmail.com,missing,1.0,2.0,11.0,11.0,0.0,11.0,11.0,-1.0,-1.0,-1.0,-1.0,11.0,0.0,-1.0,-1.0,-1.0,11.0,2.0
528,1,11502297,58.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,41.0,-1.0,missing,missing,1.0,2.0,11.0,11.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,11.0,0.0,-1.0,-1.0,-1.0,11.0,3.0
293,1,11732678,58.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,-1.0,-1.0,gmail.com,missing,1.0,1.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,63.0,0.0,-1.0,-1.0,-1.0,0.0,1.0
125,1,12360684,58.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,41.0,-1.0,gmail.com,missing,1.0,2.0,21.0,21.0,10.0,21.0,10.0,-1.0,-1.0,-1.0,-1.0,21.0,10.0,-1.0,-1.0,-1.0,21.0,4.0
278,1,12465995,58.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,41.0,-1.0,gmail.com,missing,1.0,2.0,22.0,22.0,1.0,22.0,1.0,-1.0,-1.0,-1.0,-1.0,22.0,11.0,-1.0,-1.0,-1.0,22.0,5.0
690,1,12498128,29.95,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,5.0,-1.0,gmail.com,missing,3.0,1.0,71.0,71.0,31.0,71.0,31.0,-1.0,-1.0,-1.0,-1.0,71.0,31.0,-1.0,-1.0,-1.0,71.0,5.0
322,1,12626277,49.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,-1.0,-1.0,yahoo.com,missing,1.0,1.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,-1.0,-1.0,-1.0,0.0,1.0
201,1,12691120,59.00,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,41.0,-1.0,gmail.com,missing,2.0,2.0,24.0,24.0,3.0,25.0,0.0,-1.0,-1.0,-1.0,-1.0,25.0,14.0,-1.0,-1.0,-1.0,25.0,6.0
290,1,12942370,58.95,W,17188,321.0,150.0,visa,226.0,debit,299.0,87.0,-1.0,-1.0,gmail.com,missing,188.0,168.0,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,-1.0,-1.0,-1.0,0.0,546.0


In [ ]:
# potential identifiers card 1 2 and 3 id 32 id 36 include missing id 17 too

In [ ]:
#varying address 1 varying emails repeated transactions amts .95

In [ ]:
# definitely some velocity features and tracking the amount pairs a customer has 

In [ ]:

for i in range(1, 16):
    # Some D columns might have missing values; handling NaNs is important
    df_user_check[f'D{i}_N'] = (df_user_check['TransactionDT'] / 86400) - df_user_check[f'D{i}']

# 2. Construct the user_id using the newly created D1_N
# Note: rounding D1_N to an integer or 1 decimal place helps group matches 
# that have minor floating-point differences.
# Use fillna('') on columns that might contain missing values before joining them
df_user_check['user_id'] = (
    df_user_check['card1'].fillna('').astype(str) + '_' + 
    df_user_check['card2'].fillna('').astype(str) + '_' + 
    df_user_check['card3'].fillna('').astype(str) + '_' + 
    df_user_check['addr1'].fillna('').astype(str) + '_' + 
    df_user_check['addr2'].fillna('').astype(str) + '_' + 
    df_user_check['D1_N'].round(1).astype(str))
print(df_user_check['user_id'])
